In [ ]:
!pip install pandas pyarrow joblib catboost

Looking in indexes: http://192.168.2.32:3141/root/pypi/+simple/


In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
EPS = 1e-9


def _safe_div(a, b):
    try:
        b = float(b)
        if not np.isfinite(b) or abs(b) < EPS:
            return np.nan
        return float(a) / b
    except Exception:
        return np.nan


def _to_df(events):
    if events is None:
        return pd.DataFrame()

    if isinstance(events, np.ndarray):
        events = events.tolist()

    if not isinstance(events, (list, tuple)) or len(events) == 0:
        return pd.DataFrame()

    rows = []
    for e in events:
        if isinstance(e, dict):
            rows.append(e.copy())
        else:
            try:
                rows.append(dict(e))
            except Exception:
                pass

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df.columns = [c[:-1] if isinstance(c, str) and c.endswith("_") else c for c in df.columns]

    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if "timestamp" in df.columns:
        df = df.sort_values("timestamp").reset_index(drop=True)

    return df


def _mean(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan


def _std(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.std(x)) if len(x) else np.nan


def extract_features_tiny(row):
    if isinstance(row, pd.Series):
        row = row.to_dict()
    else:
        row = dict(row)

    feats = {}

    vw = max(float(row.get("viewport_width", np.nan) or 1.0), 1.0)
    vh = max(float(row.get("viewport_height", np.nan) or 1.0), 1.0)

    mouse_df = _to_df(row.get("mouse_events", []))
    touch_df = _to_df(row.get("touch_events", []))

    mouse_total = row.get("mouse_events_total", len(mouse_df))
    touch_total = row.get("touch_events_total", len(touch_df))

    down_ts = float(row.get("pointerdown_timestamp", np.nan))
    up_ts = float(row.get("pointerup_timestamp", np.nan))
    hover_ts = float(row.get("hover_timestamp", np.nan))

    down_x = float(row.get("pointerdown_x", np.nan))
    down_y = float(row.get("pointerdown_y", np.nan))
    up_x = float(row.get("pointerup_x", np.nan))
    up_y = float(row.get("pointerup_y", np.nan))

    feats["has_mouse"] = int(len(mouse_df) > 0 or (pd.notna(mouse_total) and float(mouse_total) > 0))
    feats["has_touch"] = int(len(touch_df) > 0 or (pd.notna(touch_total) and float(touch_total) > 0))
    feats["mouse_events_total"] = float(mouse_total) if pd.notna(mouse_total) else np.nan
    feats["touch_events_total"] = float(touch_total) if pd.notna(touch_total) else np.nan

    feats["viewport_width"] = vw
    feats["viewport_height"] = vh
    feats["viewport_aspect"] = _safe_div(vw, vh)
    feats["relative_captcha_init_time"] = float(row.get("relative_captcha_init_time", np.nan))

    feats["hover_to_down_ms"] = (
        down_ts - hover_ts
        if np.isfinite(down_ts) and np.isfinite(hover_ts) and hover_ts > 0
        else np.nan
    )
    feats["down_to_up_ms"] = (
        up_ts - down_ts
        if np.isfinite(down_ts) and np.isfinite(up_ts)
        else np.nan
    )
    feats["init_to_down_ms"] = (
        down_ts - feats["relative_captcha_init_time"]
        if np.isfinite(down_ts)
        else np.nan
    )
    feats["init_to_up_ms"] = (
        up_ts - feats["relative_captcha_init_time"]
        if np.isfinite(up_ts)
        else np.nan
    )

    feats["pointerdown_x_norm"] = _safe_div(down_x, vw)
    feats["pointerdown_y_norm"] = _safe_div(down_y, vh)
    feats["pointerup_x_norm"] = _safe_div(up_x, vw)
    feats["pointerup_y_norm"] = _safe_div(up_y, vh)

    if len(touch_df) > 0 and "force" in touch_df.columns:
        feats["touch_force_mean"] = _mean(touch_df["force"].values)
        feats["touch_force_std"] = _std(touch_df["force"].values)
    else:
        feats["touch_force_mean"] = np.nan
        feats["touch_force_std"] = np.nan

    return feats


def build_features(df, n_jobs=-1, backend="loky"):
    rows = df.to_dict("records")

    feats = Parallel(n_jobs=n_jobs, backend=backend, batch_size=256, verbose=10)(
        delayed(extract_features_tiny)(r) for r in rows
    )

    return pd.DataFrame(feats)

In [ ]:
train = pd.read_parquet('train.parquet')
unlabelled = pd.read_parquet('unlabelled.parquet')
test = pd.read_parquet('test.parquet')

In [ ]:
%%time
train_x = build_features(train)
train_y = train["target"]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 256 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done 164 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done 218 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 330 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 383 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done 427 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done 472 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 519 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done 568 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done 613 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 648 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Done 685 tasks      | elapsed: 

CPU times: user 2.94 s, sys: 212 ms, total: 3.15 s
Wall time: 3.57 s


[Parallel(n_jobs=-1)]: Done 722 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 792 out of 1000 | elapsed:    3.5s remaining:    0.9s
[Parallel(n_jobs=-1)]: Done 893 out of 1000 | elapsed:    3.5s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done 994 out of 1000 | elapsed:    3.6s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:    3.6s finished


In [ ]:
%%time
test_x = build_features(test)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 256 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.6s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done 164 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 218 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done 330 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done 388 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 450 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 8960 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 17408 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 26368 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done 35328 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]: Done 44800 tasks      | 

CPU times: user 18 s, sys: 817 ms, total: 18.8 s
Wall time: 18.4 s


In [ ]:
num_cols = [c for c in train_x.columns if str(train_x[c].dtype) not in ("object", "category")]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ),
    ],
    remainder="drop",
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="liblinear",
        class_weight=None,
        random_state=42,
    )),
])

X_tr = train_x.iloc[:800, :].copy()
y_tr = train_y.iloc[:800].copy() if hasattr(train_y, "iloc") else train_y[:800]

X_val = train_x.iloc[800:, :].copy()
y_val = train_y.iloc[800:].copy() if hasattr(train_y, "iloc") else train_y[800:]

model.fit(X_tr, y_tr);

In [ ]:
scores = model.predict_proba(train_x.iloc[800:,:])[:,1]
targets = train_y.iloc[800:]
roc_auc_score(targets, scores, max_fpr=0.035)

0.5878685898205156

In [ ]:
submission = pd.DataFrame({
    'prediction': model.predict_proba(test_x)[:,1]})
submission

,prediction
0,0.202852
1,0.232637
2,0.205485
3,0.373015
4,0.191707
...,...
99995,0.217035
99996,0.403714
99997,0.152578
99998,0.222091


In [ ]:
submission.to_csv('submission.csv', index=False)